# GUI-Model: Mobile UI Transition & Action Predictor

Qwen3-VL-8B-Instruct 기반 모바일 UI 예측 모델 Fine-tuning 파이프라인.

**전체 파이프라인:**
1. **Stage 1: World Modeling** - UI State (XML) + Action → Next UI State (XML)
2. **Stage 2: Action Prediction** - Screenshot + UI State + Task → Action (JSON)
3. **Evaluation** - Test set으로 3-Way 모델 비교 평가
4. **Merge & Upload** - 학습된 모든 모델 Merge 후 HuggingFace Hub 업로드

**모델 정보:**

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Template | qwen3_vl_nothink |
| Vision Tower | Frozen |

**데이터셋 정보:**

| Dataset | Stage | MobiBench | AndroidControl |
|---------|-------|-----------|----------------|
| stage1 원본 | 1 | 3,145건 | 71,047건 |
| stage1_train | 1 | ~2,987건 (95%) | ~67,494건 (95%) |
| stage1_test | 1 | ~158건 (5%) | ~3,553건 (5%) |
| stage2 원본 | 2 | 3,147건 | 91,677건 |
| stage2_train | 2 | ~2,987건 (95%) | ~87,090건 (95%) |
| stage2_test | 2 | ~160건 (5%) | ~4,587건 (5%) |
| Images | - | 3,655개 PNG | (jsonl 내 base64/경로 참조) |

> 두 데이터셋 모두 자동 설정됩니다. Cell 3에서 `CONFIGS` 딕셔너리로 양 데이터셋의 경로, 하이퍼파라미터가 생성됩니다.

**3-Way 비교 실험:**

| Exp | Stage 1 | Stage 2 | Base Model | Config |
|-----|---------|---------|------------|--------|
| [Full World Model] | Full FT | — | `Qwen/Qwen3-VL-8B-Instruct` | stage1_full/qwen3_vl_8b_gui.yaml |
| [stage2] | — | LoRA FT | `Qwen/Qwen3-VL-8B-Instruct` | stage2_lora/base.yaml |
| [stage1+stage2] | Full FT → Merge | LoRA FT | `SaFD-00/qwen3-vl-8b-{slug}stage1-world-model` | stage2_lora/world_model.yaml |

**Base 비교:** Stage 1/Stage 2 모두 Qwen3-VL-8B-Instruct (Zero-shot) 대비 성능 측정

**학습 설정:**

| 항목 | MobiBench | AndroidControl |
|------|-----------|----------------|
| Stage 1 (Full FT) | LR=1e-5, 5 epochs, warmup=0.05 | LR=1e-5, 3 epochs, warmup=0.03 |
| Stage 2 (LoRA) | LR=3e-5, 5 epochs, warmup=0.05, r=16/α=32 | LR=5e-5, 3 epochs, warmup=0.03, r=32/α=64 |
| Stage 1 Batch / HW | 64 (2×8×4GPU), H100×4, ZeRO-3 | 동일 |
| Stage 2 Batch / HW | 32 (2×4×4GPU), H100×4 | 64 (2×8×4GPU), H100×4 |
| Eval batch (per-device) | 1 | 4 |

**평가 메트릭:**

| Metric | Stage | Description |
|--------|-------|-------------|
| eval_loss / Perplexity | 1 | Next token prediction loss |
| BLEU / ROUGE-L | 1 | 생성 XML vs GT XML 유사도 |
| Element Accuracy | 1 | XML 요소(태그+속성) 일치율 |
| Type Accuracy | 2 | Action type 일치율 |
| Bounds IoU | 2 | Bounding box 겹침 비율 |
| Overall Score | 2 | Type × (0.5×IoU + 0.5×Params) |


## Dataset Overview

본 프로젝트는 두 가지 모바일 UI 인터랙션 데이터셋을 동시에 처리합니다. Cell 3에서 두 데이터셋 모두 자동 설정됩니다.

### MobiBench

MobiBench 모바일 인터랙션 벤치마크에서 수집된 소규모 데이터셋입니다. 빠른 실험 반복에 적합합니다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 3,145건 |
| Stage 2 (Action Prediction) | 3,147건 |
| 이미지 | 3,655개 PNG |
| Train/Test Split | 95:5 (seed=42) |

**데이터 형식:**
- **Stage 1**: Screenshot + UI State (XML) + Action → Next UI State (XML)
- **Stage 2**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**Action Type 분포 (Stage 2):**

| Type | 비율 |
|------|------|
| click | ~81.9% |
| input | ~10.6% |
| swipe | ~6.3% |
| long_click | ~0.6% |
| openapp | ~0.6% |

### AndroidControl

AndroidControl 대규모 모바일 제어 데이터셋입니다. MobiBench 대비 Stage 1 ~23배, Stage 2 ~29배 대규모이며, 본 실험의 주력 데이터셋입니다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 71,047건 |
| Stage 2 (Action Prediction) | 91,677건 |
| Train/Test Split | 95:5 (seed=42) |

**데이터 형식:** MobiBench와 동일한 ShareGPT Multimodal 포맷을 사용합니다.

> **학습 레시피**: 두 데이터셋의 규모 차이를 반영해 MobiBench는 5 epochs(소규모 안정 학습), AndroidControl은 3 epochs(대규모 효율 학습)로 분기됩니다.
> Code2World, gWorld, MobileDreamer 논문 + vlm-gui-finetuning-research.md를 참고하여 설계되었습니다.


## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
import os
from dotenv import load_dotenv

# ============================================================
# === 여기서 프로젝트 루트 경로를 설정하세요 ===
BASE_DIR = "/path/to/GUI-Model"
# ============================================================

BASE_DIR = os.path.abspath(BASE_DIR)
LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")

load_dotenv(os.path.join(BASE_DIR, ".env"))

# ============================================================
# === 두 데이터셋 모두 자동 설정 (별도 전환 불필요) ===
# ============================================================
_DATASET_CONFIG = {
    "MobiBench": {
        "lf_subfolder": "GUI-Model-MB",
        "ds_prefix": "GUI-Model-MB",
        "output_prefix": "MB/",
        "hf_slug": "mb-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 5, "warmup_ratio": 0.05,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "gradient_accumulation_steps": 8,
            "per_device_eval_batch_size": 1,
        },
        "stage2": {
            "lr": "3.0e-5", "epochs": 5, "warmup_ratio": 0.05,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "gradient_accumulation_steps": 4,
            "per_device_eval_batch_size": 1,
            "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.1,
        },
    },
    "AndroidControl": {
        "lf_subfolder": "GUI-Model-AC",
        "ds_prefix": "GUI-Model-AC",
        "output_prefix": "AC/",
        "hf_slug": "ac-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "steps", "save_steps": 500,
            "eval_strategy": "steps", "eval_steps": 500,
            "gradient_accumulation_steps": 8,
            "per_device_eval_batch_size": 4,
        },
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "steps", "save_steps": 500,
            "eval_strategy": "steps", "eval_steps": 500,
            "gradient_accumulation_steps": 8,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
        },
    },
}

CONFIGS = {}
for _ds_name, _cfg in _DATASET_CONFIG.items():
    c = dict(_cfg)
    c["dataset_name"] = _ds_name
    c["data_dir"] = os.path.join(BASE_DIR, "data", _ds_name)
    c["ds_s1_train"] = f"{c['ds_prefix']}_stage1_train"
    c["ds_s1_test"]  = f"{c['ds_prefix']}_stage1_test"
    c["ds_s2_train"] = f"{c['ds_prefix']}_stage2_train"
    c["ds_s2_test"]  = f"{c['ds_prefix']}_stage2_test"
    c["hf_s1_model"]  = f"SaFD-00/qwen3-vl-8b-{c['hf_slug']}stage1-world-model"
    c["hf_s2_base"]   = f"SaFD-00/qwen3-vl-8b-{c['hf_slug']}stage2-base"
    c["hf_s2_world"]  = f"SaFD-00/qwen3-vl-8b-{c['hf_slug']}stage2-world-model"
    c["out_s1"]       = f"./outputs/{c['output_prefix']}stage1_full/full_world_model"
    c["out_s2_base"]  = f"./outputs/{c['output_prefix']}stage2_lora/lora_base"
    c["out_s2_world"] = f"./outputs/{c['output_prefix']}stage2_lora/lora_world_model"
    CONFIGS[_ds_name] = c

print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"Datasets: {list(CONFIGS.keys())}")
for _ds_name, c in CONFIGS.items():
    s1, s2 = c["stage1"], c["stage2"]
    print(f"\n--- {_ds_name} ---")
    print(f"  Data directory: {c['data_dir']}")
    print(f"  LF subfolder: {c['lf_subfolder']}")
    print(f"  HF Stage1 model: {c['hf_s1_model']}")
    print(f"  Output prefix: '{c['output_prefix']}'")
    print(f"  Stage1: {s1['epochs']} epochs, lr={s1['lr']}, warmup={s1['warmup_ratio']}, save={s1['save_strategy']}")
    print(f"  Stage2: {s2['epochs']} epochs, lr={s2['lr']}, warmup={s2['warmup_ratio']}, save={s2['save_strategy']}")


In [ ]:
%%bash
pip install -r requirements.txt

In [ ]:
%%bash
git clone https://github.com/hiyouga/LlamaFactory.git
cd LlamaFactory
pip install -e ".[torch,metrics]"
pip install -r requirements/metrics.txt -r requirements/deepspeed.txt
pip install vllm
pip install beautifulsoup4 munkres lxml

## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 Train/Test Split 후 LLaMA-Factory에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)

| File | MobiBench | AndroidControl |
|------|-----------|----------------|
| gui-model_stage1.jsonl | 3,145건 | 71,047건 |
| gui-model_stage1_train.jsonl | ~2,987건 (95%) | ~67,494건 (95%) |
| gui-model_stage1_test.jsonl | ~158건 (5%) | ~3,553건 (5%) |
| Images | 3,655개 PNG | (jsonl 내 참조) |

**Split 전략:**
- Random Split (seed=42, 재현 가능)
- 비율: 0.95 : 0.05


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

for ds_name, cfg in CONFIGS.items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Data Registration ===\n")

    for fname in ["gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    with open(DATA_PATH / "gui-model_stage1_train.jsonl", 'r') as f:
        sample = json.loads(f.readline())
    img_rel_dir = str(Path(sample['images'][0]).parent)
    img_target = LF_DATA_DIR / img_rel_dir
    src_images = DATA_PATH / "images"

    if img_target.exists() or img_target.is_symlink():
        existing_count = len(list(img_target.glob("*.png"))) if img_target.is_dir() else 0
        print(f"[Skip] Image path already exists: {img_target} ({existing_count} files)")
    else:
        img_target.parent.mkdir(parents=True, exist_ok=True)
        img_target.symlink_to(src_images.resolve())
        img_count = len(list(src_images.glob("*.png")))
        print(f"[Symlink] {img_target} -> {src_images.resolve()} ({img_count} images)")

    DATASETS_TO_REGISTER = {
        cfg["ds_s1_train"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage1_train.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        },
        cfg["ds_s1_test"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage1_test.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        }
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path

for ds_name, cfg in CONFIGS.items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Dataset Statistics ===\n")

    for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_train.jsonl", "gui-model_stage1_test.jsonl"]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            sample = entries[0] if entries else None
            msg_count = len(sample['messages']) if sample else 0
            img_count = len(sample.get('images', [])) if sample else 0
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   Messages per sample: {msg_count}")
            print(f"   Images per sample: {img_count}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
            print()

    img_dir = DATA_PATH / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.png"))
        total_size = sum(f.stat().st_size for f in imgs) / 1024 / 1024 / 1024
        print(f"  Image directory: {img_dir}")
        print(f"   Total images: {len(imgs)}")
        print(f"   Total size: {total_size:.2f} GB")

## 2. Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 **Train/Test Split** 후 LLaMA-Factory에 등록합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1과 동일 (공유)

| File | MobiBench | AndroidControl |
|------|-----------|----------------|
| gui-model_stage2.jsonl | 3,147건 | 91,677건 |
| gui-model_stage2_train.jsonl | ~2,987건 (95%) | ~87,090건 (95%) |
| gui-model_stage2_test.jsonl | ~160건 (5%) | ~4,587건 (5%) |

**Split 전략:**
- Stratified Split (action type 비율 유지)
- Random seed: 42 (재현 가능)


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

for ds_name, cfg in CONFIGS.items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Data Registration ===\n")

    for fname in ["gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    with open(DATA_PATH / "gui-model_stage2_train.jsonl", 'r') as f:
        sample = json.loads(f.readline())
    img_rel_dir = str(Path(sample['images'][0]).parent)
    img_target = LF_DATA_DIR / img_rel_dir

    if img_target.exists() or img_target.is_symlink():
        print(f"[Skip] Image path already exists: {img_target}")
    else:
        src_images = DATA_PATH / "images"
        img_target.parent.mkdir(parents=True, exist_ok=True)
        img_target.symlink_to(src_images.resolve())
        print(f"[Symlink] {img_target} -> {src_images.resolve()}")

    DATASETS_TO_REGISTER = {
        cfg["ds_s2_train"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage2_train.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        },
        cfg["ds_s2_test"]: {
            "file_name": f"../../data/{ds_name}/gui-model_stage2_test.jsonl",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS
        }
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path
from collections import Counter

for ds_name, cfg in CONFIGS.items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Dataset Statistics ===\n")

    for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

            action_types = []
            for entry in entries:
                try:
                    action = json.loads(entry['messages'][-1]['value'])
                    action_types.append(action.get('type', 'unknown'))
                except:
                    action_types.append('parse_error')

            type_counts = Counter(action_types)
            total = len(action_types)
            print(f"   Action type distribution:")
            for atype, count in type_counts.most_common():
                print(f"     {atype}: {count} ({count/total:.1%})")
            print()

In [ ]:
import json
from pathlib import Path
from collections import Counter

for ds_name, cfg in CONFIGS.items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Action Type Analysis ===")

    train_path = DATA_PATH / "gui-model_stage2_train.jsonl"
    test_path = DATA_PATH / "gui-model_stage2_test.jsonl"

    for label, fpath in [("Train", train_path), ("Test", test_path)]:
        if not fpath.exists():
            print(f"[SKIP] {label}: {fpath.name} not found")
            continue

        with open(fpath, 'r') as f:
            entries = [json.loads(line) for line in f if line.strip()]

        action_types = []
        for entry in entries:
            try:
                action = json.loads(entry['messages'][-1]['value'])
                action_types.append(action.get('type', 'unknown'))
            except:
                action_types.append('parse_error')

        type_counts = Counter(action_types)
        total = len(action_types)
        print(f"\n  {label} Set Action Types ({total} entries)")
        for atype, count in type_counts.most_common():
            print(f"    {atype}: {count} ({count/total:.1%})")

## 3. Stage 1 SFT Training

Qwen3-VL-8B-Instruct 베이스 모델을 Stage 1 World Modeling 데이터로 Full Fine-tuning합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**학습 설정:**

| 항목 | MobiBench | AndroidControl |
|------|-----------|----------------|
| Base Model | Qwen/Qwen3-VL-8B-Instruct | 동일 |
| Method | Full Fine-tuning | 동일 |
| Vision Tower | Frozen | 동일 |
| Dataset | GUI-Model-MB_stage1_train (~2,987건) | GUI-Model-AC_stage1_train (~67,494건) |
| Template | qwen3_vl_nothink | 동일 |
| Cutoff Length | 8,192 | 동일 |
| Epochs | 5 | 3 |
| Learning Rate | 1e-5 (cosine, warmup=0.05) | 1e-5 (cosine, warmup=0.03) |
| Effective Batch | 64 (2 × 8 × 4 GPU) | 동일 |
| Eval Batch (per-device) | 1 | 4 |
| Save/Eval Strategy | epoch | steps (500) |
| Weight Decay | 0.01 | 동일 |
| max_grad_norm | 1.0 | 동일 |
| Precision | bf16 | 동일 |
| DeepSpeed | ZeRO Stage 3 | 동일 |
| Hardware | H100 80GB × 4 | 동일 |
| load_best_model_at_end | true (metric=eval_loss) | 동일 |

> 학습 레시피는 Code2World, gWorld, MobileDreamer 논문 + vlm-gui-finetuning-research.md를 참고하여 설계되었습니다. AC는 데이터 규모(~67K)에 맞춰 epochs를 3으로 상향하고 `load_best_model_at_end`로 과적합을 방어합니다.


In [ ]:
from pathlib import Path

for ds_name, cfg in CONFIGS.items():
    s1 = cfg["stage1"]
    yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / "stage1_full"
    yaml_dir.mkdir(parents=True, exist_ok=True)

    # save_steps / eval_steps 는 strategy=="steps" 일 때만 출력
    save_steps_line = f"save_steps: {s1['save_steps']}\n" if s1["save_strategy"] == "steps" else ""
    eval_steps_line = f"eval_steps: {s1['eval_steps']}\n" if s1["eval_strategy"] == "steps" else ""

    STAGE1_CONFIG = f"""\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: {cfg["ds_s1_train"]}
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: {cfg["out_s1"]}
logging_steps: 1
save_strategy: {s1["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: {s1["gradient_accumulation_steps"]}
learning_rate: {s1["lr"]}
num_train_epochs: {s1["epochs"]}
lr_scheduler_type: cosine
warmup_ratio: {s1["warmup_ratio"]}
weight_decay: 0.01
max_grad_norm: 1.0
bf16: true
gradient_checkpointing: true
deepspeed: examples/deepspeed/ds_z3_config.json
# resume_from_checkpoint: true

### eval
eval_dataset: {cfg["ds_s1_test"]}
per_device_eval_batch_size: {s1["per_device_eval_batch_size"]}
eval_strategy: {s1["eval_strategy"]}
{eval_steps_line}load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false
"""

    fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
    with open(fpath, 'w') as f:
        f.write(STAGE1_CONFIG)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  dataset: {cfg['ds_s1_train']} (eval: {cfg['ds_s1_test']})")
    print(f"  output_dir: {cfg['out_s1']}")
    print(f"  epochs: {s1['epochs']}, lr: {s1['lr']}, warmup: {s1['warmup_ratio']}")
    print(f"  save: {s1['save_strategy']}" + (f"/{s1['save_steps']}" if s1['save_strategy']=='steps' else ""))
    print()


In [ ]:
os.chdir(LF_ROOT)

## [MobiBench] Stage 1 Full Fine-tuning (cosine, lr=1e-5, 5 epochs, DeepSpeed Z3) | H100 x 4
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=4 \
    llamafactory-cli train examples/custom/GUI-Model-MB/stage1_full/qwen3_vl_8b_gui.yaml

## [AndroidControl] Stage 1 Full Fine-tuning (cosine, lr=1e-5, 3 epochs, DeepSpeed Z3) | H100 x 4
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=4 \
    llamafactory-cli train examples/custom/GUI-Model-AC/stage1_full/qwen3_vl_8b_gui.yaml


## 4. Stage 1 Merge & Upload to HuggingFace

학습된 Stage 1 모델을 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**업로드 대상:**

| Dataset | HuggingFace Model ID |
|---------|---------------------|
| MobiBench | `SaFD-00/qwen3-vl-8b-stage1-world-model` |
| AndroidControl | `SaFD-00/qwen3-vl-8b-ac-stage1-world-model` |

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
from pathlib import Path

for ds_name, cfg in CONFIGS.items():
    yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / cfg["lf_subfolder"] / "gui"
    yaml_dir.mkdir(parents=True, exist_ok=True)

    MERGE_STAGE1_CONFIG = f"""\
### model
model_name_or_path: {cfg["out_s1"]}
trust_remote_code: true
template: qwen3_vl_nothink

### export
export_dir: ./exports/qwen3-vl-8b-{cfg["hf_slug"]}stage1-world-model
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {cfg["hf_s1_model"]}
"""

    fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
    with open(fpath, 'w') as f:
        f.write(MERGE_STAGE1_CONFIG)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  model: {cfg['out_s1']}")
    print(f"  hub_id: {cfg['hf_s1_model']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [MobiBench] Stage 1 Merge & HuggingFace Hub Upload
!llamafactory-cli export examples/merge_custom/GUI-Model-MB/gui/qwen3_vl_8b_gui.yaml

## [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!llamafactory-cli export examples/merge_custom/GUI-Model-AC/gui/qwen3_vl_8b_gui.yaml

## 5. Stage 1 Evaluation Pipeline

학습된 Stage 1 모델의 World Modeling 성능을 평가합니다.

**비교 대상:**

| 모델 | MobiBench | AndroidControl |
|------|-----------|----------------|
| Base (Zero-shot) | `Qwen/Qwen3-VL-8B-Instruct` | 동일 |
| Fine-tuned | `SaFD-00/qwen3-vl-8b-stage1-world-model` | `SaFD-00/qwen3-vl-8b-ac-stage1-world-model` |

### 5A. Eval-Loss (Intrinsic Evaluation)

Base(Zero-shot) 모델과 Fine-tuned 모델의 다음 토큰 예측 능력을 비교합니다.

| Metric | Description |
|--------|-------------|
| eval_loss | Test set에 대한 Cross-entropy loss |
| Perplexity | exp(eval_loss) - 낮을수록 좋음 |

### 5B. Hungarian Matching (Extrinsic Evaluation)

모델이 생성한 XML과 정답 XML을 비교합니다.

| Metric | Description |
|--------|-------------|
| BLEU-4 | n-gram 기반 텍스트 유사도 |
| ROUGE-L | Longest Common Subsequence F1 |
| Exact Match | GT XML과 완전 일치 비율 |
| Hungarian EA | Element Accuracy (매칭수 / max(pred, gt)) |
| Hungarian F1 | Precision-Recall F1 Score |
| Hungarian Prec | Precision (매칭수 / pred 요소 수) |
| Hungarian Rec | Recall (매칭수 / gt 요소 수) |
| Hungarian Text | 매칭 쌍의 Jaccard 텍스트 유사도 평균 |
| Hungarian Idx | 매칭 쌍의 index 위치 정확도 (|diff| ≤ 2) |

**테스트 데이터:**

| Dataset | Test 건수 |
|---------|----------|
| MobiBench | ~158건 (5%) |
| AndroidControl | ~3,553건 (5%) |


In [ ]:
from pathlib import Path

for ds_name, cfg in CONFIGS.items():
    yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / "stage1_eval"
    yaml_dir.mkdir(parents=True, exist_ok=True)
    (yaml_dir / "base").mkdir(exist_ok=True)
    (yaml_dir / "full_world_model").mkdir(exist_ok=True)

    s1_eval_out = f"./outputs/{cfg['output_prefix']}stage1_eval"

    BASE_EVAL_LOSS_CONFIG = f"""\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_eval: true
finetuning_type: full
freeze_vision_tower: true

### dataset
eval_dataset: {cfg["ds_s1_test"]}
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: {s1_eval_out}/eval_loss/base
overwrite_output_dir: true

### eval
per_device_eval_batch_size: 1
"""

    fpath = yaml_dir / "base" / "eval_loss.yaml"
    with open(fpath, 'w') as f:
        f.write(BASE_EVAL_LOSS_CONFIG)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

    EVAL_LOSS_CONFIG = f"""\
### model
model_name_or_path: {cfg["hf_s1_model"]}
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_eval: true
finetuning_type: full
freeze_vision_tower: true

### dataset
eval_dataset: {cfg["ds_s1_test"]}
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: {s1_eval_out}/eval_loss/full_world_model
overwrite_output_dir: true

### eval
per_device_eval_batch_size: 1
"""

    fpath = yaml_dir / "full_world_model" / "eval_loss.yaml"
    with open(fpath, 'w') as f:
        f.write(EVAL_LOSS_CONFIG)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

    BASE_PREDICT_CONFIG = f"""\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_predict: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: {cfg["ds_s1_test"]}
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: {s1_eval_out}/hungarian_matching/base
overwrite_output_dir: true

### predict
per_device_eval_batch_size: 1
predict_with_generate: true
"""

    fpath = yaml_dir / "base" / "predict.yaml"
    with open(fpath, 'w') as f:
        f.write(BASE_PREDICT_CONFIG)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

    FWM_PREDICT_CONFIG = f"""\
### model
model_name_or_path: {cfg["out_s1"]}
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_predict: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: {cfg["ds_s1_test"]}
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: {s1_eval_out}/hungarian_matching/full_world_model
overwrite_output_dir: true

### predict
per_device_eval_batch_size: 1
predict_with_generate: true
"""

    fpath = yaml_dir / "full_world_model" / "predict.yaml"
    with open(fpath, 'w') as f:
        f.write(FWM_PREDICT_CONFIG)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [MobiBench] Stage 1 - 5A. Eval-Loss: Base (Zero-shot)
!llamafactory-cli train examples/custom/GUI-Model-MB/stage1_eval/base/eval_loss.yaml

## [MobiBench] Stage 1 - 5A. Eval-Loss: Full World Model (Fine-tuned)
!llamafactory-cli train examples/custom/GUI-Model-MB/stage1_eval/full_world_model/eval_loss.yaml

## [AndroidControl] Stage 1 - 5A. Eval-Loss: Base (Zero-shot)
!llamafactory-cli train examples/custom/GUI-Model-AC/stage1_eval/base/eval_loss.yaml

## [AndroidControl] Stage 1 - 5A. Eval-Loss: Full World Model (Fine-tuned)
!llamafactory-cli train examples/custom/GUI-Model-AC/stage1_eval/full_world_model/eval_loss.yaml

In [ ]:
import json, math, os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from datetime import datetime

for ds_name, cfg in CONFIGS.items():
    op = cfg["output_prefix"]
    eval_loss_dir = Path(LF_ROOT) / "outputs" / op.rstrip("/") / "stage1_eval" / "eval_loss" if op else Path(LF_ROOT) / "outputs" / "stage1_eval" / "eval_loss"

    base_result_path = eval_loss_dir / "base" / "eval_results.json"
    fwm_result_path = eval_loss_dir / "full_world_model" / "eval_results.json"

    if not base_result_path.exists() or not fwm_result_path.exists():
        print(f"[SKIP] {ds_name}: Eval-loss results not found")
        continue

    with open(base_result_path, 'r') as f:
        base_loss_data = json.load(f)
    with open(fwm_result_path, 'r') as f:
        fwm_loss_data = json.load(f)

    base_eval_loss = base_loss_data.get('eval_loss', 0)
    base_perplexity = math.exp(base_eval_loss)
    fwm_eval_loss = fwm_loss_data.get('eval_loss', 0)
    fwm_perplexity = math.exp(fwm_eval_loss)

    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Eval-Loss ===")
    print(f"Base (Zero-shot)              eval_loss: {base_eval_loss:.4f}  perplexity: {base_perplexity:.4f}")
    print(f"Full World Model (Fine-tuned) eval_loss: {fwm_eval_loss:.4f}  perplexity: {fwm_perplexity:.4f}")

    labels = ['Base\n(Zero-shot)', 'Full World Model\n(Fine-tuned)']
    colors = ['#9E9E9E', '#FF5722']
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    losses = [base_eval_loss, fwm_eval_loss]
    bars0 = axes[0].bar(labels, losses, color=colors, width=0.5)
    axes[0].set_title('5A. Eval Loss')
    axes[0].set_ylabel('Cross-Entropy Loss')
    for bar, val in zip(bars0, losses):
        axes[0].text(bar.get_x() + bar.get_width()/2, val + max(losses)*0.02,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    axes[0].set_ylim(0, max(losses) * 1.3)
    axes[0].grid(axis='y', alpha=0.3)

    perps = [base_perplexity, fwm_perplexity]
    bars1 = axes[1].bar(labels, perps, color=colors, width=0.5)
    axes[1].set_title('5A. Perplexity')
    axes[1].set_ylabel('Perplexity = exp(loss)')
    for bar, val in zip(bars1, perps):
        axes[1].text(bar.get_x() + bar.get_width()/2, val + max(perps)*0.02,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    axes[1].set_ylim(0, max(perps) * 1.3)
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Stage 1 - 5A. Eval-Loss ({ds_name})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    img_path = eval_loss_dir / "stage1_eval_loss_evaluation.png"
    plt.savefig(img_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {img_path}")

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    loss_improve = (base_eval_loss - fwm_eval_loss) / base_eval_loss if base_eval_loss else 0
    ppl_improve = (base_perplexity - fwm_perplexity) / base_perplexity if base_perplexity else 0

    report = f"""# Stage 1 - 5A. Eval-Loss Report: Base vs Full World Model ({ds_name})

> Generated: {now}

## Experiment Setup

| Item | Value |
|------|-------|
| Dataset | {ds_name} |
| Base Model | Qwen/Qwen3-VL-8B-Instruct (Zero-shot) |
| Full World Model | {cfg["hf_s1_model"]} (Fine-tuned) |
| Test Dataset | {cfg["ds_s1_test"]} |

## Loss Metrics Comparison

| Metric | Base (Zero-shot) | Full World Model (Fine-tuned) | Improvement |
|--------|-----------------|-------------------------------|-------------|
| Eval Loss | {base_eval_loss:.4f} | {fwm_eval_loss:.4f} | {loss_improve:.1%} |
| Perplexity | {base_perplexity:.4f} | {fwm_perplexity:.4f} | {ppl_improve:.1%} |
"""

    report_path = eval_loss_dir / "evaluation_report.md"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report)
    print(f"[Saved] {report_path}")

In [ ]:
os.chdir(LF_ROOT)
_mb, _ac = CONFIGS["MobiBench"], CONFIGS["AndroidControl"]

## [MobiBench] Stage 1 - 5B. Hungarian Matching: Baseline (Qwen3-VL-8B Zero-shot)
mb_s1_hung_base = f"./outputs/{_mb['output_prefix']}stage1_eval/hungarian_matching/base"
!mkdir -p {mb_s1_hung_base}
!python scripts/vllm_infer.py \
    --model_name_or_path Qwen/Qwen3-VL-8B-Instruct \
    --dataset {_mb['ds_s1_test']} \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name {mb_s1_hung_base}/generated_predictions.jsonl \
    --matrix_save_name {mb_s1_hung_base}/predict_results.json

## [AndroidControl] Stage 1 - 5B. Hungarian Matching: Baseline (Qwen3-VL-8B Zero-shot)
ac_s1_hung_base = f"./outputs/{_ac['output_prefix']}stage1_eval/hungarian_matching/base"
!mkdir -p {ac_s1_hung_base}
!python scripts/vllm_infer.py \
    --model_name_or_path Qwen/Qwen3-VL-8B-Instruct \
    --dataset {_ac['ds_s1_test']} \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name {ac_s1_hung_base}/generated_predictions.jsonl \
    --matrix_save_name {ac_s1_hung_base}/predict_results.json

In [ ]:
os.chdir(LF_ROOT)
_mb, _ac = CONFIGS["MobiBench"], CONFIGS["AndroidControl"]

## [MobiBench] Stage 1 - 5B. Hungarian Matching: Full World Model (Fine-tuned)
mb_s1_hung_fwm = f"./outputs/{_mb['output_prefix']}stage1_eval/hungarian_matching/full_world_model"
!mkdir -p {mb_s1_hung_fwm}
!python scripts/vllm_infer.py \
    --model_name_or_path {_mb['hf_s1_model']} \
    --dataset {_mb['ds_s1_test']} \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name {mb_s1_hung_fwm}/generated_predictions.jsonl \
    --matrix_save_name {mb_s1_hung_fwm}/predict_results.json

## [AndroidControl] Stage 1 - 5B. Hungarian Matching: Full World Model (Fine-tuned)
ac_s1_hung_fwm = f"./outputs/{_ac['output_prefix']}stage1_eval/hungarian_matching/full_world_model"
!mkdir -p {ac_s1_hung_fwm}
!python scripts/vllm_infer.py \
    --model_name_or_path {_ac['hf_s1_model']} \
    --dataset {_ac['ds_s1_test']} \
    --template qwen3_vl_nothink \
    --cutoff_len 8192 \
    --image_max_pixels 4233600 \
    --enable_thinking False \
    --save_name {ac_s1_hung_fwm}/generated_predictions.jsonl \
    --matrix_save_name {ac_s1_hung_fwm}/predict_results.json

In [ ]:
import json
import re
import math
from collections import Counter
from bs4 import BeautifulSoup
from munkres import Munkres

# ── Hungarian Metric 상수 ─────────────────────────────────────────────────
INTERACTIVE_TAGS = {"button", "input", "a", "select", "textarea"}
CONTENT_TAGS     = {"p", "img", "span"}
CLICKABLE_ATTRS  = {"clickable", "long-clickable"}

W_TAG   = 3.0   # tag 불일치 패널티 (매칭 불가)
W_TEXT  = 1.5   # text 불일치
W_INDEX = 0.2   # DOM index 거리

MATCH_THRESHOLD = 1.5   # 이 이상이면 매칭 거부
INDEX_TAU       = 2     # index_acc: 차이 ≤ τ 이면 위치 정확


# ── 요소 추출 (BeautifulSoup) ─────────────────────────────────────────────

def _collect_texts(el):
    """요소 자신 + 자식 전체에서 텍스트 토큰 수집 (중복 제거, 알파벳순 join)."""
    tokens = set()
    def add(v):
        if v:
            tokens.add(v.strip())
    add(el.get("description"))
    add(el.get("id"))
    for child in el.find_all(True):
        add(child.get("description"))
        add(child.get("id"))
        t = child.get_text(strip=True)
        if t:
            tokens.add(t)
    t = el.get_text(strip=True)
    if t:
        tokens.add(t)
    return " | ".join(sorted(tokens)) if tokens else ""


def _safe_int(v, default=-1):
    try:
        return int(v)
    except (TypeError, ValueError):
        return default


def extract_elements(xml_str):
    """XML/HTML 문자열에서 interactive 요소를 평탄화하여 추출."""
    try:
        soup = BeautifulSoup(xml_str, "xml")
    except Exception:
        soup = BeautifulSoup(xml_str, "html.parser")
    elements = []
    for el in soup.find_all(True):
        tag  = el.name
        idx  = _safe_int(el.get("index", -1))
        text = _collect_texts(el)
        is_interactive = tag in INTERACTIVE_TAGS
        is_content     = (tag in CONTENT_TAGS) and bool(text)
        is_clickable   = any(el.get(a) for a in CLICKABLE_ATTRS)
        if is_interactive or is_content or is_clickable:
            elements.append({"tag": tag, "text": text, "index": idx})
    return elements


# ── 비용 함수 ─────────────────────────────────────────────────────────────

def _text_sim(a, b):
    """Jaccard 유사도 (단어 집합 기준)."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    sa = set(a.lower().replace("|", "").split())
    sb = set(b.lower().replace("|", "").split())
    return len(sa & sb) / len(sa | sb)


def _match_cost(e1, e2, max_idx):
    """pred 요소 e1과 gt 요소 e2 사이의 매칭 비용."""
    if e1["tag"] != e2["tag"]:
        return W_TAG
    tc = W_TEXT  * (1.0 - _text_sim(e1["text"], e2["text"]))
    ic = W_INDEX * (abs(e1["index"] - e2["index"]) / max(max_idx, 1))
    return round(tc + ic, 5)


# ── 헝가리안 매칭 ─────────────────────────────────────────────────────────

def _hungarian_match(pred, gt):
    """헝가리안 알고리즘으로 pred-gt 간 최적 1:1 매칭."""
    n, m = len(pred), len(gt)
    if n == 0 or m == 0:
        return [], []
    max_idx = max(
        (e["index"] for e in pred + gt if e["index"] >= 0),
        default=1,
    )
    matrix = [
        [_match_cost(p, g, max_idx) for g in gt]
        for p in pred
    ]
    size   = max(n, m)
    padded = [row + [MATCH_THRESHOLD * 2] * (size - len(row)) for row in matrix]
    while len(padded) < size:
        padded.append([MATCH_THRESHOLD * 2] * size)
    indexes = Munkres().compute(padded)
    pairs = []
    for i, j in indexes:
        if i < n and j < m and matrix[i][j] < MATCH_THRESHOLD:
            pairs.append((i, j, matrix[i][j]))
    return pairs, matrix


def compute_hungarian_acc(pred_str, gt_str):
    """모델 생성 XML과 정답 XML을 비교해 hungarian 기반 평가 메트릭을 반환."""
    _zero = {
        "hungarian_ea": 0.0, "hungarian_f1": 0.0,
        "hungarian_prec": 0.0, "hungarian_rec": 0.0,
        "hungarian_text": 0.0, "hungarian_idx": 0.0,
    }
    try:
        pred_els = extract_elements(pred_str)
        gt_els   = extract_elements(gt_str)
    except Exception:
        return _zero
    if not gt_els:
        return _zero

    pairs, _ = _hungarian_match(pred_els, gt_els)
    n_pred, n_gt, n_matched = len(pred_els), len(gt_els), len(pairs)

    ea   = n_matched / max(n_pred, n_gt) if max(n_pred, n_gt) > 0 else 0.0
    prec = n_matched / n_pred             if n_pred  > 0           else 0.0
    rec  = n_matched / n_gt               if n_gt    > 0           else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0    else 0.0

    if pairs:
        text_sims = [_text_sim(pred_els[i]["text"], gt_els[j]["text"]) for i, j, _ in pairs]
        idx_diffs = [abs(pred_els[i]["index"] - gt_els[j]["index"]) for i, j, _ in pairs]
        text_avg  = sum(text_sims) / len(text_sims)
        idx_acc   = sum(1 for d in idx_diffs if d <= INDEX_TAU) / len(idx_diffs)
    else:
        text_avg = 0.0
        idx_acc  = 0.0

    return {
        "hungarian_ea":   round(ea, 4),
        "hungarian_f1":   round(f1, 4),
        "hungarian_prec": round(prec, 4),
        "hungarian_rec":  round(rec, 4),
        "hungarian_text": round(text_avg, 4),
        "hungarian_idx":  round(idx_acc, 4),
    }


# ── 텍스트 생성 품질 메트릭 ───────────────────────────────────────────────

def calc_bleu(reference, hypothesis, max_n=4):
    """BLEU-4 score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not hyp_tokens or not ref_tokens:
        return 0.0

    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1))

        clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())

        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    return bp * math.exp(log_avg)


def calc_rouge_l(reference, hypothesis):
    """ROUGE-L (F1) score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_len = dp[m][n]
    precision = lcs_len / n
    recall = lcs_len / m

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ── Stage 1 전체 평가 함수 ────────────────────────────────────────────────

def evaluate_stage1_predictions(test_path, pred_path):
    """Stage 1 prediction 전체 평가를 수행합니다."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_text = gt_entry['messages'][-1]['value']
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))

        bleu = calc_bleu(gt_text, pred_text)
        rouge_l = calc_rouge_l(gt_text, pred_text)
        exact_match = 1.0 if gt_text.strip() == pred_text.strip() else 0.0
        hungarian = compute_hungarian_acc(pred_text, gt_text)

        results.append({
            'bleu': bleu, 'rouge_l': rouge_l,
            'exact_match': exact_match, 'hungarian': hungarian,
        })

    total = len(results)
    metrics = {
        'total': total,
        'avg_bleu': sum(r['bleu'] for r in results) / total if total else 0,
        'avg_rouge_l': sum(r['rouge_l'] for r in results) / total if total else 0,
        'exact_match_rate': sum(r['exact_match'] for r in results) / total if total else 0,
        'avg_hungarian_ea':   sum(r['hungarian']['hungarian_ea'] for r in results) / total if total else 0,
        'avg_hungarian_f1':   sum(r['hungarian']['hungarian_f1'] for r in results) / total if total else 0,
        'avg_hungarian_prec': sum(r['hungarian']['hungarian_prec'] for r in results) / total if total else 0,
        'avg_hungarian_rec':  sum(r['hungarian']['hungarian_rec'] for r in results) / total if total else 0,
        'avg_hungarian_text': sum(r['hungarian']['hungarian_text'] for r in results) / total if total else 0,
        'avg_hungarian_idx':  sum(r['hungarian']['hungarian_idx'] for r in results) / total if total else 0,
    }
    return metrics, results

print("[Loaded] Stage 1 evaluation functions: calc_bleu, calc_rouge_l, compute_hungarian_acc, evaluate_stage1_predictions")

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path

all_stage1_metrics = {}

for ds_name, cfg in CONFIGS.items():
    op = cfg["output_prefix"]
    s1_eval_dir = Path(LF_ROOT) / "outputs" / op.rstrip("/") / "stage1_eval" if op else Path(LF_ROOT) / "outputs" / "stage1_eval"
    test_path = Path(cfg["data_dir"]) / "gui-model_stage1_test.jsonl"

    MODEL_PREDS = {
        "Baseline (Zero-shot)": s1_eval_dir / "hungarian_matching" / "base" / "generated_predictions.jsonl",
        "Full World Model (Fine-tuned)": s1_eval_dir / "hungarian_matching" / "full_world_model" / "generated_predictions.jsonl",
    }

    print(f"\n{'='*70}")
    print(f"=== {ds_name} Stage 1 Evaluation ===")

    base_loss_path = s1_eval_dir / "eval_loss" / "base" / "eval_results.json"
    fwm_loss_path = s1_eval_dir / "eval_loss" / "full_world_model" / "eval_results.json"

    if base_loss_path.exists() and fwm_loss_path.exists():
        with open(base_loss_path, 'r') as f:
            base_loss_metrics = json.load(f)
        with open(fwm_loss_path, 'r') as f:
            fwm_loss_metrics = json.load(f)
        print(f"\n  Loss-based Evaluation:")
        print(f"    Base     eval_loss: {base_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(base_loss_metrics['eval_loss']):.4f}")
        print(f"    FWM      eval_loss: {fwm_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(fwm_loss_metrics['eval_loss']):.4f}")
    else:
        print("  [SKIP] Loss evaluation results not found")

    ds_metrics = {}
    for model_name, pred_path in MODEL_PREDS.items():
        if not pred_path.exists():
            print(f"  [SKIP] {model_name}: Prediction results not found")
            continue
        metrics, results = evaluate_stage1_predictions(str(test_path), str(pred_path))
        ds_metrics[model_name] = metrics

        print(f"\n  === {model_name} - Generation Evaluation ===")
        summary = pd.DataFrame({
            'Metric': ['BLEU-4', 'ROUGE-L', 'Exact Match', 'Hungarian EA', 'Hungarian F1',
                       'Hungarian Prec', 'Hungarian Rec', 'Hungarian Text', 'Hungarian Idx'],
            'Score': [f"{metrics['avg_bleu']:.4f}", f"{metrics['avg_rouge_l']:.4f}",
                      f"{metrics['exact_match_rate']:.1%}", f"{metrics['avg_hungarian_ea']:.4f}",
                      f"{metrics['avg_hungarian_f1']:.4f}", f"{metrics['avg_hungarian_prec']:.4f}",
                      f"{metrics['avg_hungarian_rec']:.4f}", f"{metrics['avg_hungarian_text']:.4f}",
                      f"{metrics['avg_hungarian_idx']:.4f}"]
        })
        display(summary)

    all_stage1_metrics[ds_name] = ds_metrics

    if len(ds_metrics) > 1:
        print(f"\n  === {ds_name} Stage 1 Comparison ===")
        comparison = pd.DataFrame({
            name: {
                'BLEU-4': f"{m['avg_bleu']:.4f}", 'ROUGE-L': f"{m['avg_rouge_l']:.4f}",
                'Exact Match': f"{m['exact_match_rate']:.1%}", 'Hungarian EA': f"{m['avg_hungarian_ea']:.4f}",
                'Hungarian F1': f"{m['avg_hungarian_f1']:.4f}", 'Hungarian Prec': f"{m['avg_hungarian_prec']:.4f}",
                'Hungarian Rec': f"{m['avg_hungarian_rec']:.4f}", 'Hungarian Text': f"{m['avg_hungarian_text']:.4f}",
                'Hungarian Idx': f"{m['avg_hungarian_idx']:.4f}",
            } for name, m in ds_metrics.items()
        })
        display(comparison)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for ds_name, ds_metrics in all_stage1_metrics.items():
    if not ds_metrics:
        continue
    cfg = CONFIGS[ds_name]
    model_names = list(ds_metrics.keys())
    n_models = len(model_names)
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    text_metrics = ['avg_bleu', 'avg_rouge_l', 'exact_match_rate']
    text_labels = ['BLEU-4', 'ROUGE-L', 'Exact Match']
    x1 = np.arange(len(text_metrics))
    width = 0.8 / n_models

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in text_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x1 + offset, values, width, label=name, color=colors[i])

    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title('5B. Text Generation Metrics')
    axes[0].set_xticks(x1); axes[0].set_xticklabels(text_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    hung_metrics = ['avg_hungarian_ea', 'avg_hungarian_f1', 'avg_hungarian_prec',
                    'avg_hungarian_rec', 'avg_hungarian_text', 'avg_hungarian_idx']
    hung_labels = ['EA', 'F1', 'Precision', 'Recall', 'Text Sim', 'Index Acc']
    x2 = np.arange(len(hung_metrics))

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in hung_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])

    axes[1].set_xlabel('Metric'); axes[1].set_ylabel('Score')
    axes[1].set_title('5B. Hungarian Element Matching')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(hung_labels, rotation=15)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Stage 1 Evaluation ({ds_name})', fontsize=14, fontweight='bold')
    plt.tight_layout()

    op = cfg["output_prefix"]
    s1_eval_dir = Path(LF_ROOT) / "outputs" / op.rstrip("/") / "stage1_eval" if op else Path(LF_ROOT) / "outputs" / "stage1_eval"
    save_path = str(s1_eval_dir / "hungarian_matching" / "stage1_hungarian_matching_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
import json, math
from pathlib import Path
from datetime import datetime

def generate_stage1_eval_report(model_name, metrics, output_dir, ds_name, cfg, loss_metrics=None, comparison_metrics=None):
    m = metrics
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = []
    lines.append(f"# Stage 1 - 5B. Hungarian Matching Report: {model_name} ({ds_name})")
    lines.append(f"\n> Generated: {now}\n")
    lines.append("## Experiment Setup\n")
    lines.append("| Item | Value |")
    lines.append("|------|-------|")
    lines.append(f"| Dataset | {ds_name} |")
    lines.append(f"| Model | {model_name} |")
    lines.append(f"| Test Samples | {m['total']} |")
    lines.append(f"| Training | {cfg['stage1_epochs']} epochs, LR={cfg['stage1_lr']} (cosine) |")
    lines.append("")
    if loss_metrics:
        eval_loss = loss_metrics.get('eval_loss', 0)
        lines.append("## Loss-based Metrics\n")
        lines.append("| Metric | Score |")
        lines.append("|--------|-------|")
        lines.append(f"| Eval Loss | {eval_loss:.4f} |")
        lines.append(f"| Perplexity | {math.exp(eval_loss):.4f} |")
        lines.append("")
    lines.append("## Overall Metrics\n")
    lines.append("| Metric | Score |")
    lines.append("|--------|-------|")
    for label, key, fmt in [("BLEU-4","avg_bleu",".4f"),("ROUGE-L","avg_rouge_l",".4f"),
        ("Exact Match","exact_match_rate",".1%"),("Hungarian EA","avg_hungarian_ea",".4f"),
        ("Hungarian F1","avg_hungarian_f1",".4f"),("Hungarian Prec","avg_hungarian_prec",".4f"),
        ("Hungarian Rec","avg_hungarian_rec",".4f"),("Hungarian Text","avg_hungarian_text",".4f"),
        ("Hungarian Idx","avg_hungarian_idx",".4f")]:
        lines.append(f"| {label} | {m[key]:{fmt}} |")
    lines.append("")
    if comparison_metrics:
        lines.append("## Model Comparison\n")
        lines.append("| Metric | " + " | ".join(comparison_metrics.keys()) + " |")
        lines.append("|--------|" + "|".join(["-------"] * len(comparison_metrics)) + "|")
        for label, key, fmt in [("BLEU-4","avg_bleu",".4f"),("ROUGE-L","avg_rouge_l",".4f"),
            ("Exact Match","exact_match_rate",".1%"),("Hungarian EA","avg_hungarian_ea",".4f"),
            ("Hungarian F1","avg_hungarian_f1",".4f"),("Hungarian Prec","avg_hungarian_prec",".4f"),
            ("Hungarian Rec","avg_hungarian_rec",".4f"),("Hungarian Text","avg_hungarian_text",".4f"),
            ("Hungarian Idx","avg_hungarian_idx",".4f")]:
            values = []
            scores = [cm[key] for cm in comparison_metrics.values()]
            best = max(scores)
            for name, cm in comparison_metrics.items():
                val = cm[key]
                formatted = f"{val:{fmt}}"
                if val == best and len(set(scores)) > 1:
                    formatted = f"**{formatted}**"
                values.append(formatted)
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        lines.append("")
    report = "\n".join(lines)
    output_path = Path(output_dir) / "evaluation_report.md"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)
    return str(output_path)

for ds_name, ds_metrics in all_stage1_metrics.items():
    cfg = CONFIGS[ds_name]
    op = cfg["output_prefix"]
    s1_eval_dir = Path(LF_ROOT) / "outputs" / op.rstrip("/") / "stage1_eval" if op else Path(LF_ROOT) / "outputs" / "stage1_eval"
    OUTPUT_DIRS = {
        "Baseline (Zero-shot)": s1_eval_dir / "hungarian_matching" / "base",
        "Full World Model (Fine-tuned)": s1_eval_dir / "hungarian_matching" / "full_world_model",
    }
    loss_metrics_map = {}
    for lbl, lp in [("Baseline (Zero-shot)", "base"), ("Full World Model (Fine-tuned)", "full_world_model")]:
        p = s1_eval_dir / "eval_loss" / lp / "eval_results.json"
        if p.exists():
            with open(p, 'r') as f:
                loss_metrics_map[lbl] = json.load(f)
    for model_name, m in ds_metrics.items():
        output_dir = OUTPUT_DIRS.get(model_name)
        if output_dir:
            saved = generate_stage1_eval_report(model_name, m, output_dir, ds_name, cfg,
                                                 loss_metrics=loss_metrics_map.get(model_name),
                                                 comparison_metrics=ds_metrics)
            print(f"[Saved] {saved}")
print("\nDone.")

## 6. Stage 2 SFT Training

**핵심 가설**: GUI World Modeling으로 사전학습된 모델은 동일 베이스 대비 Action Prediction SFT에서 더 높은 성능을 보인다.

| ID | 역할 | 모델 | MobiBench HF ID | AndroidControl HF ID |
|----|------|------|-----------------|---------------------|
| [stage2] | Base | Qwen3-VL-8B-Instruct | `Qwen/Qwen3-VL-8B-Instruct` | 동일 |
| [stage1+stage2] | World Model | Stage 1 Merged | `SaFD-00/qwen3-vl-8b-stage1-world-model` | `SaFD-00/qwen3-vl-8b-ac-stage1-world-model` |

**Base 비교:** Qwen3-VL-8B-Instruct (Zero-shot, 학습 없음)도 함께 평가

**핵심 비교:**
- stage2 vs stage1+stage2 → World Modeling 사전학습이 Action Prediction에 미치는 효과

**공통 학습 설정:**

| 항목 | MobiBench | AndroidControl |
|------|-----------|----------------|
| Method | LoRA (r=16, α=32, dropout=0.1) | LoRA (r=32, α=64, dropout=0.1) |
| Dataset | GUI-Model-MB_stage2_train (~2,987건) | GUI-Model-AC_stage2_train (~87,090건) |
| Effective Batch | 32 (2 × 4 × 4 GPU) | 64 (2 × 8 × 4 GPU) |
| Eval Batch (per-device) | 1 | 4 |
| Learning Rate | 3e-5 (cosine, warmup=0.05) | 5e-5 (cosine, warmup=0.03) |
| Epochs | 5 | 3 |
| Save/Eval Strategy | epoch | steps (500) |
| Weight Decay | 0.01 | 동일 |
| max_grad_norm | 1.0 | 동일 |
| Precision | bf16 | 동일 |
| Hardware | H100 80GB × 4 | 동일 |
| load_best_model_at_end | true (metric=eval_loss) | 동일 |

> AC LoRA rank/alpha 상향(r=16→32, α=32→64)은 50K-100K 샘플 규모에 맞춘 capacity 확장입니다 (vlm-gui-finetuning-research.md 권장 64-128의 보수 선택).


In [ ]:
import os
from pathlib import Path

for ds_name, cfg in CONFIGS.items():
    s2 = cfg["stage2"]
    yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / "stage2_lora"
    yaml_dir.mkdir(parents=True, exist_ok=True)

    save_steps_line = f"save_steps: {s2['save_steps']}\n" if s2["save_strategy"] == "steps" else ""
    eval_steps_line = f"eval_steps: {s2['eval_steps']}\n" if s2["eval_strategy"] == "steps" else ""

    COMMON_CONFIG = f"""\
### model
model_name_or_path: {{model_name_or_path}}
trust_remote_code: true
image_max_pixels: 4233600

### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: true
lora_rank: {s2["lora_rank"]}
lora_alpha: {s2["lora_alpha"]}
lora_target: all
lora_dropout: {s2["lora_dropout"]}

### dataset
dataset: {cfg["ds_s2_train"]}
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/{{output_dir}}
logging_steps: 1
save_strategy: {s2["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 2
gradient_accumulation_steps: {s2["gradient_accumulation_steps"]}
learning_rate: {s2["lr"]}
num_train_epochs: {s2["epochs"]}
lr_scheduler_type: cosine
warmup_ratio: {s2["warmup_ratio"]}
weight_decay: 0.01
max_grad_norm: 1.0
bf16: true
gradient_checkpointing: true
# resume_from_checkpoint: true

### eval
eval_dataset: {cfg["ds_s2_test"]}
per_device_eval_batch_size: {s2["per_device_eval_batch_size"]}
eval_strategy: {s2["eval_strategy"]}
{eval_steps_line}load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false
"""

    MODELS = {
        "base.yaml": {
            "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
            "output_dir": f"{cfg['output_prefix']}stage2_lora/lora_base",
        },
        "world_model.yaml": {
            "model_name_or_path": cfg["hf_s1_model"],
            "output_dir": f"{cfg['output_prefix']}stage2_lora/lora_world_model",
        },
    }

    print(f"=== {ds_name} Stage 2 YAML ===")
    for fname, params in MODELS.items():
        content = COMMON_CONFIG.format(
            model_name_or_path=params["model_name_or_path"],
            output_dir=params["output_dir"],
        )
        fpath = yaml_dir / fname
        with open(fpath, 'w') as f:
            f.write(content)
        print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
        print(f"  base: {params['model_name_or_path']}")
        print(f"  output: ./outputs/{params['output_dir']}")
        print(f"  epochs: {s2['epochs']}, lr: {s2['lr']}, warmup: {s2['warmup_ratio']}")
        print(f"  LoRA: r={s2['lora_rank']}, α={s2['lora_alpha']}, dropout={s2['lora_dropout']}")
        print()


In [ ]:
os.chdir(LF_ROOT)

## [MobiBench] [stage2] Stage 2 - Base (Qwen3-VL-8B-Instruct) | H100 x 4
!llamafactory-cli train examples/custom/GUI-Model-MB/stage2_lora/base.yaml

## [AndroidControl] [stage2] Stage 2 - Base (Qwen3-VL-8B-Instruct) | H100 x 4
!llamafactory-cli train examples/custom/GUI-Model-AC/stage2_lora/base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [MobiBench] [stage1+stage2] Stage 2 - World Model (Stage 1 Merged) | H100 x 4
!llamafactory-cli train examples/custom/GUI-Model-MB/stage2_lora/world_model.yaml

## [AndroidControl] [stage1+stage2] Stage 2 - World Model (Stage 1 Merged) | H100 x 4
!llamafactory-cli train examples/custom/GUI-Model-AC/stage2_lora/world_model.yaml

## 7. Stage 2 Merge & Upload to HuggingFace

Stage 2 LoRA 어댑터를 베이스 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**Merge 대상 (MobiBench):**

| ID | Base Model | Export |
|----|------------|--------|
| [stage2] | `Qwen/Qwen3-VL-8B-Instruct` | `SaFD-00/qwen3-vl-8b-stage2-base` |
| [stage1+stage2] | `SaFD-00/qwen3-vl-8b-stage1-world-model` | `SaFD-00/qwen3-vl-8b-stage2-world-model` |

**Merge 대상 (AndroidControl):**

| ID | Base Model | Export |
|----|------------|--------|
| [stage2] | `Qwen/Qwen3-VL-8B-Instruct` | `SaFD-00/qwen3-vl-8b-ac-stage2-base` |
| [stage1+stage2] | `SaFD-00/qwen3-vl-8b-ac-stage1-world-model` | `SaFD-00/qwen3-vl-8b-ac-stage2-world-model` |

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
from pathlib import Path

for ds_name, cfg in CONFIGS.items():
    yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / cfg["lf_subfolder"] / "stage2"
    yaml_dir.mkdir(parents=True, exist_ok=True)

    MERGE_TEMPLATE = """\
### model
model_name_or_path: {model_name_or_path}
adapter_name_or_path: {adapter_path}
trust_remote_code: true
finetuning_type: lora
template: qwen3_vl_nothink

### export
export_dir: ./exports/{export_dir}
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {hub_model_id}
"""

    MERGE_MODELS = {
        "merge_base.yaml": {
            "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
            "adapter_path": f"./outputs/{cfg['output_prefix']}stage2_lora/lora_base",
            "export_dir": f"qwen3-vl-8b-{cfg['hf_slug']}stage2-base",
            "hub_model_id": cfg["hf_s2_base"],
        },
        "merge_world_model.yaml": {
            "model_name_or_path": cfg["hf_s1_model"],
            "adapter_path": f"./outputs/{cfg['output_prefix']}stage2_lora/lora_world_model",
            "export_dir": f"qwen3-vl-8b-{cfg['hf_slug']}stage2-world-model",
            "hub_model_id": cfg["hf_s2_world"],
        },
    }

    print(f"=== {ds_name} Stage 2 Merge YAML ===")
    for fname, params in MERGE_MODELS.items():
        content = MERGE_TEMPLATE.format(**params)
        fpath = yaml_dir / fname
        with open(fpath, 'w') as f:
            f.write(content)
        print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
        print(f"  base: {params['model_name_or_path']}")
        print(f"  adapter: {params['adapter_path']}")
        print(f"  export: {params['hub_model_id']}")
        print()

In [ ]:
os.chdir(LF_ROOT)

## [MobiBench] [stage2] Merge - Base (Qwen3-VL-8B-Instruct)
!llamafactory-cli export examples/merge_custom/GUI-Model-MB/stage2/merge_base.yaml

## [AndroidControl] [stage2] Merge - Base (Qwen3-VL-8B-Instruct)
!llamafactory-cli export examples/merge_custom/GUI-Model-AC/stage2/merge_base.yaml

In [ ]:
os.chdir(LF_ROOT)

## [MobiBench] [stage1+stage2] Merge - World Model (Stage 1 Merged + Stage 2 LoRA)
!llamafactory-cli export examples/merge_custom/GUI-Model-MB/stage2/merge_world_model.yaml

## [AndroidControl] [stage1+stage2] Merge - World Model (Stage 1 Merged + Stage 2 LoRA)
!llamafactory-cli export examples/merge_custom/GUI-Model-AC/stage2/merge_world_model.yaml

## 8. Stage 2 Evaluation Pipeline

Test set을 사용하여 3개 모델(stage2, stage1+stage2 + Base Zero-shot)의 Action Prediction 성능을 비교합니다.

**파이프라인:**
1. Prediction YAML 생성 (2모델 + base zero-shot)
2. `vllm_infer.py`로 test set prediction 수행
3. 평가 함수로 action 단위 메트릭 계산
4. 모델별 비교 테이블 & 시각화

**평가 메트릭:**

| Metric | Formula | Description |
|--------|---------|-------------|
| Parse Rate | 유효 JSON / 전체 | 모델 출력 파싱 성공률 |
| Type Accuracy | 정확 type / 전체 | Action type 일치율 |
| Bounds IoU | IoU(GT, Pred) | Bounding box 겹침 비율 |
| Params Match | 정확 params / 전체 | Action parameters 일치율 |
| Overall Score | Type × (0.5×IoU + 0.5×Params) | 종합 점수 |

**테스트 데이터:**

| Dataset | Test 건수 |
|---------|----------|
| MobiBench | ~160건 (5%) |
| AndroidControl | ~4,587건 (5%) |


In [ ]:
os.chdir(LF_ROOT)
_mb, _ac = CONFIGS["MobiBench"], CONFIGS["AndroidControl"]

## [MobiBench] [Base Zero-shot] Prediction
mb_s2_eval_base = f"./outputs/{_mb['output_prefix']}stage2_eval/base"
!mkdir -p {mb_s2_eval_base}
!python scripts/vllm_infer.py \
    --model_name_or_path Qwen/Qwen3-VL-8B-Instruct \
    --dataset {_mb['ds_s2_test']} \
    --template qwen3_vl_nothink --cutoff_len 8192 --image_max_pixels 4233600 --enable_thinking False \
    --save_name {mb_s2_eval_base}/generated_predictions.jsonl \
    --matrix_save_name {mb_s2_eval_base}/predict_results.json

## [AndroidControl] [Base Zero-shot] Prediction
ac_s2_eval_base = f"./outputs/{_ac['output_prefix']}stage2_eval/base"
!mkdir -p {ac_s2_eval_base}
!python scripts/vllm_infer.py \
    --model_name_or_path Qwen/Qwen3-VL-8B-Instruct \
    --dataset {_ac['ds_s2_test']} \
    --template qwen3_vl_nothink --cutoff_len 8192 --image_max_pixels 4233600 --enable_thinking False \
    --save_name {ac_s2_eval_base}/generated_predictions.jsonl \
    --matrix_save_name {ac_s2_eval_base}/predict_results.json

In [ ]:
os.chdir(LF_ROOT)
_mb, _ac = CONFIGS["MobiBench"], CONFIGS["AndroidControl"]

## [MobiBench] [stage2] Prediction - Base (Qwen3-VL-8B-Instruct + LoRA)
mb_s2_eval_lora_base = f"./outputs/{_mb['output_prefix']}stage2_eval/lora_base"
!mkdir -p {mb_s2_eval_lora_base}
!python scripts/vllm_infer.py \
    --model_name_or_path {_mb['hf_s2_base']} \
    --dataset {_mb['ds_s2_test']} \
    --template qwen3_vl_nothink --cutoff_len 8192 --image_max_pixels 4233600 --enable_thinking False \
    --save_name {mb_s2_eval_lora_base}/generated_predictions.jsonl \
    --matrix_save_name {mb_s2_eval_lora_base}/predict_results.json

## [AndroidControl] [stage2] Prediction - Base (Qwen3-VL-8B-Instruct + LoRA)
ac_s2_eval_lora_base = f"./outputs/{_ac['output_prefix']}stage2_eval/lora_base"
!mkdir -p {ac_s2_eval_lora_base}
!python scripts/vllm_infer.py \
    --model_name_or_path {_ac['hf_s2_base']} \
    --dataset {_ac['ds_s2_test']} \
    --template qwen3_vl_nothink --cutoff_len 8192 --image_max_pixels 4233600 --enable_thinking False \
    --save_name {ac_s2_eval_lora_base}/generated_predictions.jsonl \
    --matrix_save_name {ac_s2_eval_lora_base}/predict_results.json

In [ ]:
os.chdir(LF_ROOT)
_mb, _ac = CONFIGS["MobiBench"], CONFIGS["AndroidControl"]

## [MobiBench] [stage1+stage2] Prediction - World Model (Stage 1 Merged + LoRA)
mb_s2_eval_lora_wm = f"./outputs/{_mb['output_prefix']}stage2_eval/lora_world_model"
!mkdir -p {mb_s2_eval_lora_wm}
!python scripts/vllm_infer.py \
    --model_name_or_path {_mb['hf_s2_world']} \
    --dataset {_mb['ds_s2_test']} \
    --template qwen3_vl_nothink --cutoff_len 8192 --image_max_pixels 4233600 --enable_thinking False \
    --save_name {mb_s2_eval_lora_wm}/generated_predictions.jsonl \
    --matrix_save_name {mb_s2_eval_lora_wm}/predict_results.json

## [AndroidControl] [stage1+stage2] Prediction - World Model (Stage 1 Merged + LoRA)
ac_s2_eval_lora_wm = f"./outputs/{_ac['output_prefix']}stage2_eval/lora_world_model"
!mkdir -p {ac_s2_eval_lora_wm}
!python scripts/vllm_infer.py \
    --model_name_or_path {_ac['hf_s2_world']} \
    --dataset {_ac['ds_s2_test']} \
    --template qwen3_vl_nothink --cutoff_len 8192 --image_max_pixels 4233600 --enable_thinking False \
    --save_name {ac_s2_eval_lora_wm}/generated_predictions.jsonl \
    --matrix_save_name {ac_s2_eval_lora_wm}/predict_results.json

In [ ]:
import json
import re
from collections import defaultdict

def parse_bounds(bounds_str):
    """Parse bounds string '[x1, y1, x2, y2]' to list of floats."""
    try:
        nums = re.findall(r'[\d.]+', str(bounds_str))
        return [float(n) for n in nums[:4]] if len(nums) >= 4 else None
    except:
        return None

def calc_iou(box1, box2):
    """Calculate IoU between two bounding boxes [x1, y1, x2, y2]."""
    if box1 is None or box2 is None:
        return 0.0
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def parse_action(text):
    """Parse action JSON from model output text."""
    text = text.strip()
    # Try direct JSON parse
    try:
        return json.loads(text)
    except:
        pass
    # Try extracting JSON from markdown code block
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            pass
    # Try finding first JSON object
    match = re.search(r'\{[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    return None

def evaluate_single(gt_action, pred_action):
    """Evaluate a single prediction against ground truth."""
    result = {'type_correct': False, 'iou': 0.0, 'params_correct': False, 'has_bounds': False, 'has_params': False}

    if pred_action is None:
        return result

    gt_type = gt_action.get('type', '')
    pred_type = pred_action.get('type', '')
    result['type_correct'] = (gt_type.lower() == pred_type.lower())

    # Bounds IoU
    gt_bounds = parse_bounds(gt_action.get('bounds'))
    pred_bounds = parse_bounds(pred_action.get('bounds'))
    if gt_bounds is not None:
        result['has_bounds'] = True
        result['iou'] = calc_iou(gt_bounds, pred_bounds)

    # Params match
    param_keys = {'openapp': 'app', 'input': 'text', 'swipe': 'direction'}
    if gt_type.lower() in param_keys:
        result['has_params'] = True
        key = param_keys[gt_type.lower()]
        gt_val = str(gt_action.get(key, '')).strip().lower()
        pred_val = str(pred_action.get(key, '')).strip().lower()
        result['params_correct'] = (gt_val == pred_val)
    elif gt_type.lower() not in ['click', 'long click']:
        # Types without bounds or special params
        result['has_params'] = False

    return result

def evaluate_predictions(test_path, pred_path):
    """Evaluate all predictions and return metrics."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]

    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    per_type = defaultdict(lambda: {'count': 0, 'type_correct': 0, 'iou_sum': 0.0, 'iou_count': 0, 'params_correct': 0, 'params_count': 0})

    for i, (gt_entry, pred_entry) in enumerate(zip(gt_entries, pred_entries)):
        gt_action = json.loads(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))
        pred_action = parse_action(pred_text)

        r = evaluate_single(gt_action, pred_action)
        r['gt_type'] = gt_action.get('type', 'unknown').lower()
        results.append(r)

        t = r['gt_type']
        per_type[t]['count'] += 1
        per_type[t]['type_correct'] += int(r['type_correct'])
        if r['has_bounds']:
            per_type[t]['iou_sum'] += r['iou']
            per_type[t]['iou_count'] += 1
        if r['has_params']:
            per_type[t]['params_correct'] += int(r['params_correct'])
            per_type[t]['params_count'] += 1

    total = len(results)
    parsed = sum(1 for r in results if r['type_correct'] or r.get('gt_type'))
    type_correct = sum(r['type_correct'] for r in results)
    bounds_entries = [r for r in results if r['has_bounds']]
    params_entries = [r for r in results if r['has_params']]

    avg_iou = sum(r['iou'] for r in results) / total if total else 0
    cond_iou = sum(r['iou'] for r in bounds_entries) / len(bounds_entries) if bounds_entries else 0
    params_acc = sum(r['params_correct'] for r in params_entries) / len(params_entries) if params_entries else 0
    cond_params = params_acc
    type_acc = type_correct / total if total else 0
    overall = type_acc * (0.5 * avg_iou + 0.5 * (sum(r['params_correct'] for r in results) / total if total else 0))

    per_type_summary = {}
    for t, d in per_type.items():
        per_type_summary[t] = {
            'count': d['count'],
            'type_acc': d['type_correct'] / d['count'] if d['count'] else 0,
            'avg_iou': d['iou_sum'] / d['iou_count'] if d['iou_count'] else 0,
            'params_acc': d['params_correct'] / d['params_count'] if d['params_count'] else 0,
        }

    metrics = {
        'total': total,
        'parse_rate': parsed / total if total else 0,
        'type_accuracy': type_acc,
        'avg_bounds_iou': avg_iou,
        'cond_bounds_iou': cond_iou,
        'params_accuracy': sum(r['params_correct'] for r in results) / total if total else 0,
        'cond_params_acc': cond_params,
        'overall_score': overall,
        'per_type': per_type_summary,
    }

    return metrics, results

print("[Loaded] Evaluation functions: parse_bounds, calc_iou, parse_action, evaluate_single, evaluate_predictions")

In [ ]:
import json
import pandas as pd
from pathlib import Path

all_s2_metrics = {}
all_s2_results = {}

for ds_name, cfg in CONFIGS.items():
    op = cfg["output_prefix"]
    s2_eval_dir = Path(LF_ROOT) / "outputs" / op.rstrip("/") / "stage2_eval" if op else Path(LF_ROOT) / "outputs" / "stage2_eval"
    test_path = Path(cfg["data_dir"]) / "gui-model_stage2_test.jsonl"

    MODEL_PREDS = {
        "Base (Zero-shot)": s2_eval_dir / "base" / "generated_predictions.jsonl",
        "stage2": s2_eval_dir / "lora_base" / "generated_predictions.jsonl",
        "stage1+stage2": s2_eval_dir / "lora_world_model" / "generated_predictions.jsonl",
            }

    ds_metrics = {}
    ds_results = {}
    for model_name, pred_path in MODEL_PREDS.items():
        if not pred_path.exists():
            print(f"[SKIP] {ds_name}/{model_name}: {pred_path.name} not found")
            continue
        metrics, results = evaluate_predictions(str(test_path), str(pred_path))
        ds_metrics[model_name] = metrics
        ds_results[model_name] = results
        print(f"[Done] {ds_name}/{model_name}")

    all_s2_metrics[ds_name] = ds_metrics
    all_s2_results[ds_name] = ds_results

    if ds_metrics:
        summary_df = pd.DataFrame({
            name: {
                'Parse Rate': f"{m['parse_rate']:.1%}", 'Type Accuracy': f"{m['type_accuracy']:.1%}",
                'Bounds IoU': f"{m['avg_bounds_iou']:.3f}", 'Params Accuracy': f"{m['params_accuracy']:.1%}",
                'Cond. IoU': f"{m['cond_bounds_iou']:.3f}", 'Cond. Params': f"{m['cond_params_acc']:.1%}",
                'Overall Score': f"{m['overall_score']:.3f}",
            } for name, m in ds_metrics.items()
        }).T
        print(f"\n{'='*70}")
        print(f"=== Stage 2 Model Comparison - {ds_name} (3-Way) ===")
        print("="*70)
        display(summary_df)
        for model_name, m in ds_metrics.items():
            print(f"\n--- {ds_name}/{model_name} Per-Type ---")
            type_df = pd.DataFrame(m['per_type']).T
            type_df.columns = ['Count', 'Type Acc', 'Avg IoU', 'Params Acc']
            display(type_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for ds_name, ds_metrics in all_s2_metrics.items():
    if not ds_metrics:
        continue
    cfg = CONFIGS[ds_name]
    model_names = list(ds_metrics.keys())
    metrics_to_plot = ['type_accuracy', 'avg_bounds_iou', 'params_accuracy', 'overall_score']
    metric_labels = ['Type Accuracy', 'Bounds IoU', 'Params Accuracy', 'Overall Score']
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    x = np.arange(len(metrics_to_plot))
    n_models = len(model_names)
    width = 0.8 / n_models
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]
    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in metrics_to_plot]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x + offset, values, width, label=name, color=colors[i])
    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title(f'Stage 2: 3-Way Model Comparison ({ds_name})')
    axes[0].set_xticks(x); axes[0].set_xticklabels(metric_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    action_types = sorted(set(t for m in ds_metrics.values() for t in m['per_type'].keys()))
    x2 = np.arange(len(action_types))
    for i, name in enumerate(model_names):
        values = [ds_metrics[name]['per_type'].get(t, {}).get('type_acc', 0) for t in action_types]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])
    axes[1].set_xlabel('Action Type'); axes[1].set_ylabel('Type Accuracy')
    axes[1].set_title(f'Per-Type Accuracy Comparison ({ds_name})')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(action_types, rotation=30, fontsize=8)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    op = cfg["output_prefix"]
    s2_eval_dir = Path(LF_ROOT) / "outputs" / op.rstrip("/") / "stage2_eval" if op else Path(LF_ROOT) / "outputs" / "stage2_eval"
    save_path = str(s2_eval_dir / "stage2_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
import json
from pathlib import Path
from datetime import datetime

def generate_eval_report(model_name, metrics, results, output_dir, ds_name, cfg, comparison_metrics=None):
    m = metrics
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = []
    lines.append(f"# Stage 2 Evaluation Report: {model_name} ({ds_name})")
    lines.append(f"\n> Generated: {now}\n")
    lines.append("## Experiment Setup\n")
    lines.append("| Item | Value |")
    lines.append("|------|-------|")
    lines.append(f"| Dataset | {ds_name} |")
    lines.append(f"| Model | {model_name} |")
    lines.append(f"| Test Samples | {m['total']} |")
    lines.append(f"| Method | LoRA (r=16, a=32, dropout=0.1) |")
    lines.append(f"| Training | {cfg['stage2_epochs']} epochs, LR={cfg['stage2_lr']} (cosine) |")
    lines.append("")
    lines.append("## Overall Metrics\n")
    lines.append("| Metric | Score |")
    lines.append("|--------|-------|")
    lines.append(f"| Parse Rate | {m['parse_rate']:.1%} |")
    lines.append(f"| Type Accuracy | {m['type_accuracy']:.1%} |")
    lines.append(f"| Bounds IoU (avg) | {m['avg_bounds_iou']:.4f} |")
    lines.append(f"| Bounds IoU (cond.) | {m['cond_bounds_iou']:.4f} |")
    lines.append(f"| Params Accuracy (avg) | {m['params_accuracy']:.1%} |")
    lines.append(f"| Params Accuracy (cond.) | {m['cond_params_acc']:.1%} |")
    lines.append(f"| **Overall Score** | **{m['overall_score']:.4f}** |")
    lines.append("")
    lines.append("## Per-Type Breakdown\n")
    lines.append("| Action Type | Count | Type Acc | Avg IoU | Params Acc |")
    lines.append("|-------------|-------|----------|---------|------------|")
    for t, d in sorted(m['per_type'].items(), key=lambda x: -x[1]['count']):
        lines.append(f"| {t} | {d['count']} | {d['type_acc']:.1%} | {d['avg_iou']:.4f} | {d['params_acc']:.1%} |")
    lines.append("")
    if comparison_metrics:
        lines.append("## Model Comparison\n")
        lines.append("| Metric | " + " | ".join(comparison_metrics.keys()) + " |")
        lines.append("|--------|" + "|".join(["-------"] * len(comparison_metrics)) + "|")
        for label, key, fmt in [("Parse Rate","parse_rate",".1%"),("Type Accuracy","type_accuracy",".1%"),
            ("Bounds IoU","avg_bounds_iou",".4f"),("Cond. IoU","cond_bounds_iou",".4f"),
            ("Params Accuracy","params_accuracy",".1%"),("Cond. Params","cond_params_acc",".1%"),
            ("Overall Score","overall_score",".4f")]:
            values = []
            scores = [cm[key] for cm in comparison_metrics.values()]
            best = max(scores)
            for name, cm in comparison_metrics.items():
                val = cm[key]
                formatted = f"{val:{fmt}}"
                if val == best and len(set(scores)) > 1:
                    formatted = f"**{formatted}**"
                values.append(formatted)
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        lines.append("")
    report = "\n".join(lines)
    output_path = Path(output_dir) / "evaluation_report.md"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)
    return str(output_path)

for ds_name, ds_metrics in all_s2_metrics.items():
    cfg = CONFIGS[ds_name]
    op = cfg["output_prefix"]
    s2_eval_dir = Path(LF_ROOT) / "outputs" / op.rstrip("/") / "stage2_eval" if op else Path(LF_ROOT) / "outputs" / "stage2_eval"
    OUTPUT_DIRS = {
        "Base (Zero-shot)": s2_eval_dir / "base",
        "stage2": s2_eval_dir / "lora_base",
        "stage1+stage2": s2_eval_dir / "lora_world_model",
    }
    for model_name, m in ds_metrics.items():
        output_dir = OUTPUT_DIRS.get(model_name)
        if output_dir:
            saved = generate_eval_report(model_name, m, all_s2_results[ds_name][model_name],
                                          output_dir, ds_name, cfg, comparison_metrics=ds_metrics)
            print(f"[Saved] {saved}")
print("\nDone.")